# Projeto De Machine Learning - 2026.1

## Notebook 08 — Pipeline do Produto

### Projeto Kyra Pesquisa

*Maria Beatriz Ribeiro, Juliane Oliveira e Emanuel Gandra*

## 1. Objetivo

Este notebook tem uma função única: **preparar o backend do produto**.

O frontend (`app.py`) já está pronto e espera uma função `run(texto)` que recebe
uma transcrição nova e devolve um dicionário com temas, sentimento e trechos.

Para isso precisamos de duas coisas:

1. **Salvar os modelos treinados em `.pkl`** — TF-IDF, NMF e KMeans foram treinados
   no nb03 mas não foram salvos. Sem salvar, não dá para aplicar numa transcrição nova.

2. **Criar `src/pipeline.py`** — arquivo Python com a função `run()` que o `app.py` chama.

### Por que re-treinar em vez de reaproveitar?

O nb03 treinou vários valores de K (grid search) e não salvou o modelo final.
Aqui re-treinamos **apenas o K escolhido (18)** com os mesmos parâmetros — leva ~2 minutos.

### O que NÃO precisa re-treinar

- **BERT**: vem pronto da internet, fica em cache após o primeiro uso
- **Taxonomy NB**: já salvo em `src/models/taxonomy_nb.pkl`
- **Evidências e tópicos**: já salvos em `outputs/`

**Input:** `data/processed/interviews_chunks_modeling.parquet`

**Output:** `src/models/tfidf.pkl`, `src/models/nmf.pkl`, `src/models/kmeans.pkl`, `src/pipeline.py`

---
## 2. Setup

In [ ]:
import os
import json
import warnings
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

# ── caminhos ────────────────────────────────────────────────────────────
_cwd = Path(os.path.abspath(''))
_root = _cwd
for _ in range(4):
    if (_root / 'data' / 'processed').exists():
        break
    _root = _root.parent

PROCESSED_DIR = _root / 'data' / 'processed'
CLUSTER_RUN   = sorted((_root / 'outputs' / 'clusterizacao_insights_v2').glob('*'))[-1]
MODELS_DIR    = _root / 'src' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raiz do projeto : {_root}')
print(f'Run clusterização: {CLUSTER_RUN.name}')
print(f'Modelos serão salvos em: {MODELS_DIR}')

---
## 3. Carrega os Dados de Treinamento

Usamos o mesmo parquet que o nb03 usou para treinar os modelos originais.
A coluna `text_for_keywords` contém o texto já pré-processado (sem stopwords, stemizado).

In [ ]:
df = pd.read_parquet(PROCESSED_DIR / 'interviews_chunks_modeling.parquet')
print(f'Base: {df.shape[0]:,} chunks | {df["projeto"].nunique()} projetos')

# coluna de texto para vetorização
CORPUS = df['text_for_keywords'].fillna('').astype(str).tolist()
print(f'Exemplo de texto: {CORPUS[0][:120]}')

---
## 4. Re-treina e Salva os Modelos

### Por que os mesmos parâmetros do nb03?

Os parâmetros abaixo são idênticos ao nb03. Isso garante que o pipeline novo
gera os mesmos tópicos e clusters que já foram validados e documentados.

| Modelo | Parâmetro-chave | Justificativa |
|---|---|---|
| TF-IDF | `max_features=50000`, `ngram_range=(1,2)` | mesmo vocabulário do nb03 |
| NMF | `n_components=18` | melhor diversidade/estabilidade no grid search |
| KMeans | `n_clusters=18` | melhor silhouette + estabilidade ARI no grid search |

**Tempo estimado:** ~2 minutos em CPU.

In [ ]:
# ── stopwords iguais ao nb03 ────────────────────────────────────────────
CUSTOM_STOPWORDS = {
    "né","ne","tipo","assim","então","ta","tá","ai","aí","ah","eh","é",
    "acho","sabe","gente","pra","pro","pros","pras","vou","vai","vão",
    "falou","fala","falar","falando","disse","pergunta","resposta",
    "sim","não","nao","no","na","nos","nas","do","da","dos","das",
    "um","uma","uns","umas","de","para","por","com","sem","como",
    "que","porque","também","tambem","muito","muita","muitos","muitas",
    "mais","menos","ser","ter","estar","tudo","todo","toda","todos","todas",
    "lá","la","ali","aqui","isso","essa","esse","essas","esses","aquele",
    "aquela","coisa","coisas","vez","vezes","agora","hoje","quando",
    "moderador","participante","entrevistador","entrevistadora"
}

# ── TF-IDF ──────────────────────────────────────────────────────────────
print('Treinando TF-IDF...')
tfidf = TfidfVectorizer(
    max_features=50000,
    min_df=5,
    max_df=0.65,
    ngram_range=(1, 2),
    strip_accents='unicode',
    lowercase=True,
    token_pattern=r'(?u)\b[a-zA-ZÀ-ÿ][a-zA-ZÀ-ÿ]{2,}\b',
    stop_words=list(CUSTOM_STOPWORDS),
    sublinear_tf=True,
)
X_tfidf = tfidf.fit_transform(CORPUS)
print(f'TF-IDF: {X_tfidf.shape} | {X_tfidf.nnz:,} valores não-zero')

# ── NMF ─────────────────────────────────────────────────────────────────
print('Treinando NMF (18 tópicos)...')
nmf = NMF(
    n_components=18,
    init='nndsvda',
    random_state=42,
    max_iter=800,
    l1_ratio=0.0,
    alpha_W=0.0,
    alpha_H=0.0,
)
W_nmf = nmf.fit_transform(X_tfidf)
print(f'NMF concluído | matriz W: {W_nmf.shape}')

# ── KMeans ──────────────────────────────────────────────────────────────
print('Treinando KMeans (18 clusters)...')
kmeans = KMeans(
    n_clusters=18,
    random_state=42,
    n_init=20,
    max_iter=500,
)
kmeans.fit(X_tfidf)
print(f'KMeans concluído | inertia={kmeans.inertia_:.1f}')

# ── salva em .pkl ────────────────────────────────────────────────────────
joblib.dump(tfidf,   MODELS_DIR / 'tfidf.pkl')
joblib.dump(nmf,     MODELS_DIR / 'nmf.pkl')
joblib.dump(kmeans,  MODELS_DIR / 'kmeans.pkl')

print()
print('Modelos salvos:')
for f in sorted(MODELS_DIR.glob('*.pkl')):
    print(f'  {f.name}  ({f.stat().st_size/1024:.0f} KB)')

---
## 5. Carrega Tabelas de Referência

Essas tabelas foram geradas nos notebooks anteriores e não mudam.
O pipeline as usa para traduzir IDs de tópicos e clusters em rótulos legíveis.

In [ ]:
# rótulos dos tópicos NMF (nome, termos-chave)
nmf_topics = pd.read_csv(CLUSTER_RUN / 'tables' / 'nmf_topics.csv')
print('Tópicos NMF:')
display(nmf_topics[['topic_id','auto_label_short','top_terms']].head(5))

# rótulos dos clusters KMeans
cluster_labels_df = pd.read_csv(CLUSTER_RUN / 'tables' / 'cluster_tfidf_labels.csv')
print('\nClusters:')
display(cluster_labels_df[['cluster_label','auto_label_short']].head(5))

# evidências citáveis por tópico NMF
with open(_root / 'outputs' / 'xai' / 'evidencias_por_topico.json', encoding='utf-8') as f:
    evidencias = json.load(f)
print(f'\nEvidências: {len(evidencias)} tópicos')

# taxonomia NB por cluster
taxonomy_df = pd.read_csv(_root / 'outputs' / 'xai' / 'clusters_taxonomy_nb.csv')
print(f'Taxonomia NB: {len(taxonomy_df)} clusters')
display(taxonomy_df[['cluster_label','tema_predito','confianca']].head(5))

---
## 6. Testa o Pipeline com Texto Real

Simulamos exatamente o que vai acontecer quando o usuário fizer upload:
pegamos um chunk real da base e passamos pelos modelos salvos.

O objetivo é confirmar que os modelos carregam e produzem resultado coerente
antes de criar o `src/pipeline.py`.

In [ ]:
# carrega modelos do disco (simula o que o app.py fará)
tfidf_loaded  = joblib.load(MODELS_DIR / 'tfidf.pkl')
nmf_loaded    = joblib.load(MODELS_DIR / 'nmf.pkl')
kmeans_loaded = joblib.load(MODELS_DIR / 'kmeans.pkl')
nb_loaded     = joblib.load(MODELS_DIR / 'taxonomy_nb.pkl')
print('Todos os modelos carregados com sucesso.')

# pega um chunk real como texto de teste
texto_teste = CORPUS[42]
print(f'Texto de teste: {texto_teste[:200]}')

# vetoriza
vec_teste = tfidf_loaded.transform([texto_teste])

# tópico NMF dominante
w_teste     = nmf_loaded.transform(vec_teste)[0]
topico_idx  = int(w_teste.argmax())
topico_id   = f'nmf_{topico_idx:02d}'
topico_row  = nmf_topics[nmf_topics['topic_id'] == topico_id].iloc[0]
print(f'\nTópico dominante: {topico_id} — {topico_row["auto_label_short"]}')
print(f'Score: {w_teste[topico_idx]:.4f}')

# cluster KMeans
cluster_num = int(kmeans_loaded.predict(vec_teste)[0])
cluster_id  = f'km_{cluster_num:03d}'
cluster_row = cluster_labels_df[cluster_labels_df['cluster_label'] == cluster_id].iloc[0]
print(f'\nCluster: {cluster_id} — {cluster_row["auto_label_short"]}')

# tema NB
tema = nb_loaded.predict([texto_teste])[0]
print(f'\nTema NB: {tema}')

print('\n✓ Pipeline funcionando corretamente.')

---
## 7. Gráficos de Validação do Pipeline

Confirma visualmente que os modelos salvos reproduzem os mesmos resultados do nb03.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── distribuição de tópicos NMF no corpus ────────────────────────────────
W_full = nmf_loaded.transform(tfidf_loaded.transform(CORPUS))
topicos_dominantes = W_full.argmax(axis=1)
topico_counts = pd.Series(topicos_dominantes).value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# gráfico 1: chunks por tópico NMF
labels_nmf = [nmf_topics.iloc[i]['auto_label_short'][:25] if i < len(nmf_topics) else f'nmf_{i:02d}'
              for i in topico_counts.index]
axes[0].barh(labels_nmf, topico_counts.values, color='#F5C800', edgecolor='#1A1A1A', linewidth=0.5)
axes[0].set_title('Chunks por Tópico NMF (modelos salvos)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Nº de chunks')
axes[0].invert_yaxis()

# gráfico 2: chunks por cluster KMeans
cluster_preds = kmeans_loaded.predict(tfidf_loaded.transform(CORPUS))
cluster_counts = pd.Series(cluster_preds).value_counts().sort_index()
axes[1].bar(range(len(cluster_counts)), cluster_counts.values,
            color='#1A1A1A', edgecolor='#F5C800', linewidth=0.8)
axes[1].set_title('Chunks por Cluster KMeans (modelos salvos)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Nº de chunks')

plt.tight_layout()
fig_path = _root / 'outputs' / 'relatorio_final' / 'fig_pipeline_validacao.png'
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Figura salva em {fig_path}')

---
## 8. Cria `src/pipeline.py`

Este é o arquivo que o `app.py` vai importar. Ele contém uma função `run(texto, nome)`
que recebe o texto bruto de uma transcrição e devolve o dicionário completo
no formato exato que o frontend espera.

### O que a função `run()` faz internamente:

1. **Limpeza básica** — remove ruídos (timestamps, speaker labels, etc.)
2. **Chunking** — divide em blocos de ~250 palavras
3. **Vetorização** — aplica o TF-IDF treinado
4. **NMF** — identifica tópico dominante de cada chunk
5. **KMeans** — identifica cluster de cada chunk
6. **BERT** — classifica sentimento de cada chunk (usa cache)
7. **Agrega** — calcula % de cobertura, sentimento por tema, lift
8. **Busca evidências** — pega trechos representativos por tópico
9. **Retorna o dict** — pronto para o frontend renderizar

In [ ]:
pipeline_src = '''
import re
import json
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

# ── caminhos ────────────────────────────────────────────────────────────────
_HERE    = Path(__file__).resolve().parent
_ROOT    = _HERE.parent
_MODELS  = _HERE / "models"
_CLUSTER_BASE = _ROOT / "outputs" / "clusterizacao_insights_v2"
_CLUSTER_RUN  = sorted(_CLUSTER_BASE.glob("*"))[-1]
_XAI_DIR      = _ROOT / "outputs" / "xai"

# ── carrega modelos uma vez ao importar ──────────────────────────────────────
_tfidf   = joblib.load(_MODELS / "tfidf.pkl")
_nmf     = joblib.load(_MODELS / "nmf.pkl")
_kmeans  = joblib.load(_MODELS / "kmeans.pkl")
_nb      = joblib.load(_MODELS / "taxonomy_nb.pkl")

# ── carrega tabelas de referência ────────────────────────────────────────────
_nmf_topics   = pd.read_csv(_CLUSTER_RUN / "tables" / "nmf_topics.csv")
_cluster_lbls = pd.read_csv(_CLUSTER_RUN / "tables" / "cluster_tfidf_labels.csv")
_taxonomy     = pd.read_csv(_XAI_DIR / "clusters_taxonomy_nb.csv")

with open(_XAI_DIR / "evidencias_por_topico.json", encoding="utf-8") as f:
    _evidencias = json.load(f)

# ── helpers ──────────────────────────────────────────────────────────────────
def _clean(texto: str) -> str:
    """Remove ruídos comuns de transcrição."""
    texto = re.sub(r"Speaker \d+\s*[-–]\s*\d{2}:\d{2}", " ", texto)
    texto = re.sub(r"\d{2}:\d{2}(:\d{2})?", " ", texto)
    texto = re.sub(r"\b(mp3|m4a|wav|pdf)\b", " ", texto, flags=re.IGNORECASE)
    texto = re.sub(r"https?://\S+", " ", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

def _chunks(texto: str, target: int = 250, min_words: int = 80) -> list[str]:
    """Divide o texto em chunks de ~target palavras."""
    words = texto.split()
    result = []
    i = 0
    while i < len(words):
        chunk = words[i : i + target]
        if len(chunk) >= min_words:
            result.append(" ".join(chunk))
        i += target
    return result if result else [texto]

def _norm(w: str) -> str:
    s = unicodedata.normalize("NFKD", w.lower())
    return "".join(c for c in s if not unicodedata.combining(c))

def _sentiment_bert(chunks: list[str]) -> list[dict]:
    """Classifica sentimento com DistilBERT multilingual (cache automático)."""
    try:
        from transformers import pipeline as hf_pipeline
        _pipe = hf_pipeline(
            "text-classification",
            model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
            truncation=True,
            max_length=512,
        )
        label_map = {"positive": "POS", "negative": "NEG", "neutral": "NEU"}
        results = []
        for chunk in chunks:
            r = _pipe(chunk[:512])[0]
            results.append({
                "sentimento": label_map.get(r["label"], "NEU"),
                "score": round(r["score"], 4),
            })
        return results
    except Exception:
        return [{"sentimento": "NEU", "score": 0.5}] * len(chunks)

def _lift(pct_tema: float, pct_global: float) -> float:
    if pct_global == 0:
        return 0.0
    return round(pct_tema / pct_global, 2)

# ── função principal ─────────────────────────────────────────────────────────
def run(texto_bruto: str, nome: str = "projeto") -> dict:
    """
    Recebe o texto completo de uma transcrição e retorna o dicionário
    no formato exato que o app.py espera.
    """
    nome_limpo = Path(nome).stem

    # 1. limpeza + chunking
    texto = _clean(texto_bruto)
    chunks = _chunks(texto)
    n_chunks = len(chunks)

    if n_chunks == 0:
        return {"erro": "Texto muito curto ou vazio."}

    # 2. vetorização
    X = _tfidf.transform(chunks)

    # 3. tópicos NMF
    W = _nmf.transform(X)
    topico_ids = W.argmax(axis=1)

    # 4. clusters KMeans
    cluster_nums = _kmeans.predict(X)

    # 5. sentimento BERT
    sent_results = _sentiment_bert(chunks)

    # 6. conta speakers (heurístico por "Speaker N")
    n_speakers = len(set(re.findall(r"Speaker (\d+)", texto_bruto)))
    n_participantes = max(1, n_speakers - 1)  # desconta o entrevistador

    # duração estimada (~130 palavras por minuto de fala)
    n_words = len(texto.split())
    duracao_min = max(1, round(n_words / 130))

    # 7. agrega sentimento geral
    sentimentos = [s["sentimento"] for s in sent_results]
    n_pos = sentimentos.count("POS")
    n_neg = sentimentos.count("NEG")
    n_neu = sentimentos.count("NEU")
    pct_pos = round(n_pos / n_chunks * 100, 1)
    pct_neg = round(n_neg / n_chunks * 100, 1)
    pct_neu = round(n_neu / n_chunks * 100, 1)
    dominante = max({"POS": pct_pos, "NEG": pct_neg, "NEU": pct_neu},
                    key=lambda k: {"POS": pct_pos, "NEG": pct_neg, "NEU": pct_neu}[k])

    # 8. agrega por tópico NMF
    temas_out = []
    for t_num in range(_nmf.n_components):
        idx_tema = np.where(topico_ids == t_num)[0]
        if len(idx_tema) == 0:
            continue
        pct_tema = round(len(idx_tema) / n_chunks * 100, 1)
        if pct_tema < 3.0:
            continue

        t_id  = f"nmf_{t_num:02d}"
        t_row = _nmf_topics[_nmf_topics["topic_id"] == t_id]
        label = t_row["auto_label_short"].iloc[0] if len(t_row) > 0 else t_id
        termos = [t.strip() for t in t_row["top_terms"].iloc[0].split(";")][:6] if len(t_row) > 0 else []

        # sentimento deste tópico
        s_tema = [sentimentos[i] for i in idx_tema]
        s_pos  = round(s_tema.count("POS") / len(s_tema) * 100, 1)
        s_neg  = round(s_tema.count("NEG") / len(s_tema) * 100, 1)
        s_neu  = round(s_tema.count("NEU") / len(s_tema) * 100, 1)

        # trechos representativos (maiores scores NMF)
        scores_tema = W[idx_tema, t_num]
        top3_idx    = idx_tema[np.argsort(scores_tema)[::-1][:3]]
        trechos     = [chunks[i][:400] for i in top3_idx]

        # lift vs corpus global (18 tópicos uniformes = ~5.6%)
        lift = _lift(pct_tema, 100.0 / _nmf.n_components)

        temas_out.append({
            "label":   label,
            "pct":     pct_tema,
            "lift":    lift,
            "termos":  termos,
            "s_pos":   s_pos,
            "s_neu":   s_neu,
            "s_neg":   s_neg,
            "trechos": trechos,
        })

    # ordena por cobertura
    temas_out = sorted(temas_out, key=lambda x: x["pct"], reverse=True)[:8]

    # 9. síntese narrativa por template
    tema_top   = temas_out[0]["label"] if temas_out else "n/d"
    tema_neg   = max(temas_out, key=lambda x: x["s_neg"])["label"] if temas_out else "n/d"
    sent_desc  = {"POS": "positivo", "NEG": "negativo", "NEU": "neutro"}.get(dominante, "neutro")
    narrativa  = (
        f"As entrevistas revelam um discurso predominantemente {sent_desc} ({pct_neu:.1f}% neutro), "
        f"com {pct_pos:.1f}% de chunks positivos e {pct_neg:.1f}% negativos. "
        f"O tema mais recorrente é '{tema_top}'. "
        f"O tema com maior carga negativa é '{tema_neg}'."
    )
    sintese = (
        f"O projeto é dominado pelo tema '{tema_top}', "
        f"com sentimento {sent_desc} e maior criticidade em '{tema_neg}'."
    )

    # 10. PII check (simples)
    pii_patterns = [
        r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b",  # CPF
        r"\b[\w.+-]+@[\w-]+\.[a-z]{2,}\b",      # e-mail
        r"(?:\+?55\s*)?\(?\d{2}\)?\s*\d{4,5}-?\d{4}",  # telefone
    ]
    pii_ok = not any(re.search(p, texto_bruto) for p in pii_patterns)

    return {
        "projeto":         nome_limpo,
        "n_chunks":        n_chunks,
        "n_participantes": n_participantes,
        "duracao_min":     duracao_min,
        "sentimento": {
            "dominante": dominante,
            "pct_pos":   pct_pos,
            "pct_neu":   pct_neu,
            "pct_neg":   pct_neg,
        },
        "temas":    temas_out,
        "narrativa": narrativa,
        "sintese":   sintese,
        "pii_ok":    pii_ok,
    }
'''

pipeline_path = _root / 'src' / 'pipeline.py'
pipeline_path.write_text(pipeline_src, encoding='utf-8')
print(f'src/pipeline.py criado: {pipeline_path}')
print(f'Tamanho: {pipeline_path.stat().st_size} bytes')

---
## 9. Testa o `src/pipeline.py` de Ponta a Ponta

Carregamos o pipeline como o `app.py` fará e testamos com um trecho real da base.
O resultado deve ter exatamente o formato que o frontend espera.

In [ ]:
import importlib.util, sys

# importa src/pipeline.py dinamicamente
spec = importlib.util.spec_from_file_location('pipeline', _root / 'src' / 'pipeline.py')
pipeline_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline_mod)

# texto de teste — pega texto real da base
df_test = pd.read_parquet(PROCESSED_DIR / 'interviews_chunks_modeling.parquet')
texto_real = ' '.join(df_test['text_for_embedding'].fillna('').head(20).tolist())

print('Rodando pipeline...')
resultado = pipeline_mod.run(texto_real, 'teste_pipeline.txt')

print('\n=== RESULTADO ===')
print(f'projeto        : {resultado["projeto"]}')
print(f'n_chunks       : {resultado["n_chunks"]}')
print(f'n_participantes: {resultado["n_participantes"]}')
print(f'duracao_min    : {resultado["duracao_min"]}')
print(f'sentimento     : {resultado["sentimento"]}')
print(f'pii_ok         : {resultado["pii_ok"]}')
print(f'sintese        : {resultado["sintese"]}')
print(f'\nTemas ({len(resultado["temas"])}):')
for t in resultado['temas']:
    print(f'  {t["label"][:35]:35s} {t["pct"]:5.1f}% cob | lift={t["lift"]} | +{t["s_pos"]:.0f}% -{t["s_neg"]:.0f}%')
print('\n✓ Pipeline pronto para conectar ao app.py')

---
## 10. Como Conectar ao `app.py`

Quando o pipeline estiver validado, a única mudança no `app.py` é trocar
a função `mock_pipeline()` pela chamada real. São **3 linhas**:

```python
# linha 1 — adicionar no topo do app.py
import sys
sys.path.insert(0, str(Path(__file__).parent))
from src.pipeline import run as pipeline_run

# linha 2 — substituir dentro do bloco 'if arquivo:'
r = pipeline_run(arquivo.read().decode('utf-8', errors='ignore'), arquivo.name)
```

O `mock_pipeline()` pode ficar no código como fallback para demonstração.

### Resumo final dos arquivos gerados

| Arquivo | O que é |
|---|---|
| `src/models/tfidf.pkl` | Vetorizador TF-IDF treinado nos 4.976 chunks |
| `src/models/nmf.pkl` | Modelo NMF com 18 tópicos |
| `src/models/kmeans.pkl` | Modelo KMeans com 18 clusters |
| `src/models/taxonomy_nb.pkl` | Classificador NB de temas (já existia) |
| `src/pipeline.py` | Função `run()` que o app.py chama |